In [1]:
import numpy as np
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import gc

In [2]:
class OneBitOptimizer(torch.optim.Optimizer):
    def __init__(self, params, lr=0.001, momentum=0.9):
        defaults = dict(lr=lr, momentum=momentum)
        super().__init__(params, defaults)
        self.basis_mats = {}
        for group in self.param_groups:
            for p in group['params']:
                row, col = p.shape
                n = max(2, int(2 ** np.ceil(np.log2(max(row, col))))) # need to find what dim we want for basis since not always power of 2 or square
                if n in self.basis_mats:
                    basis = self.basis_mats[n]
                else:
                    basis = self.orthogonal_basis(n)
                    self.basis_mats[n] = basis
                w_padded = np.zeros((n, n), dtype=np.float32)
                w_padded[:row, :col] = p.data.cpu().numpy()
                c_raw = self.encode(w_padded, basis)
                c = self.binarize(c_raw)
                self.state[p]["binary_coeffs"] = c
                self.state[p]["momentum_buffer"] = np.zeros((n, n), dtype=np.float32)
                w_dec_padded = self.decode(c, basis)
                w_dec = w_dec_padded[:row, :col]
                p.data.copy_(torch.tensor(w_dec, dtype=p.dtype, device=p.device))
    
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                c = self.state[p]["binary_coeffs"]
                m = self.state[p]["momentum_buffer"]
                row, col = p.shape
                grad_padded = np.zeros((m.shape[0], m.shape[0]), dtype=np.float32)
                grad_padded[:row, :col] = p.grad.data.cpu().numpy()
                m = group["momentum"] * m + (1 - group["momentum"]) * grad_padded
                bits_to_update = 10
                for _ in range(bits_to_update):
                    c = self.coordinate_descent_step(c, m, self.basis_mats[m.shape[0]])
                self.state[p]["binary_coeffs"] = c
                w_dec_padded = self.decode(c, self.basis_mats[m.shape[0]])
                w_dec = w_dec_padded[:row, :col]
                p.data.copy_(torch.tensor(w_dec, dtype=p.dtype, device=p.device))
        return loss
        
    def orthogonal_basis(self, n):
        if (n <= 0 or np.log2(n) != round(np.log2(n))):
            raise ValueError("n is not a power of 2")        
        iters = np.log2(n)
        bm1 = np.array([[-1, 1], [1, 1]])
        bm2 = np.array([[1, -1], [1, 1]])
        bm3 = np.array([[1, 1], [-1, 1]])
        bm4 = np.array([[1, 1], [1, -1]]) 
        two_matrices = [bm1, bm2, bm3, bm4]
        matrices = two_matrices
        for _ in range(int(iters) - 1):
            new_matrices = []
            for mat1 in two_matrices:
                for mat2 in matrices:
                    new_matrices.append(np.kron(mat1, mat2))
            matrices = new_matrices
        return matrices
    
    def encode(self, w, basis):
        c = []
        for mat in basis:
            c.append(np.trace(mat.T @ w)/(w.shape[0]**2)) # need to normalize since basis isnt orthonormal
        return c

    def decode(self, c, basis):
        w = np.zeros_like(basis[0], dtype=np.float64)
        for i in range(len(basis)):
            w += basis[i] * c[i]
        return w

    def binarize(self, c):
        for i in range(len(c)):
            if (c[i] >= 0):
                c[i] = 1
            else:
                c[i] = -1
        return c

    def coordinate_descent_step(self, c, m, basis):
        scores = []
        for i in range(len(c)):
            scores.append(np.trace(basis[i].T @ m) * c[i])
        flip_index = 0
        for i in range(len(scores)):
            if scores[i] < scores[flip_index]:
                flip_index = i
        if scores[flip_index] >= 0:
            return c
        c[flip_index] *= -1
        return c

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

In [ ]:
class mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.layers(x)

model = mlp()
criterion = nn.CrossEntropyLoss()
optimizer = OneBitOptimizer(model.parameters())
gc.collect()

In [ ]:
epochs = 10
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

In [ ]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs}]')
    print("Training loss", (running_loss / total))  
    train_losses.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss_eval / total))
    test_losses.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)
    
    gc.collect()